In [ ]:
import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
)

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)


In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "MANIOBRALLEGADA",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ORIGEN",
            "SALIDA",
            "MANIOBRASALIDA",
            "MANIOBRA",
        ]
    )
}
from src.processor import  SitraProcessor

In [ ]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool,
    JCTC: bool,
    pro: bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones, trenes=trenes, inicio=start_date, fin=end_date, pro=pro,xSIV=xSIV, jCTC=JCTC
    )
    historico = historico[
        (historico["Fecha"] >= pd.to_datetime(start_date))
        & (historico["Fecha"] <= pd.to_datetime(end_date))
    ]
    # Usamos movimientos auditados
    #historico = historico[
         #np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
     #]
    historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
        subset=["Movimiento"]
    )
    historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico

In [ ]:

ntrenes = [rellenarId(f"{i}") for i in np.arange(100000)]
start_date = "2025-06-30"
end_date = "2025-07-01"
historico_pro = cargarHistorico(start_date,end_date,[],ntrenes,xSIV=True,JCTC=False,pro=True)

In [ ]:
folder_ru = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Sitra\sitra_26_05\RuOperation")
folder_rm = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Sitra\sitra_26_05\realmovement")
logs_ru = []
logs_rm = []
for fname in folder_ru.glob("*.csv"):
    with fname.open("r", encoding="utf-8") as f:
        data = f.read()
        data = regex.sub(r'^"timestamp","message"\n?', "", data)
        data = regex.sub(r" +", "", data)
        if regex.search(r'("{4}|""[\w\d\.-]+?"")', data):
            print(data)
            data = regex.sub(r'""', '"', data)
        lines = data.split("\n")
        logs_ru.extend([
            regex.split(r"<\?xml.+?\?>", l.strip('"'))[-1]
            for l in lines if l.strip()
        ])
for fname in folder_rm.glob("*.csv"):
    with fname.open("r", encoding="utf-8") as f:
        data = f.read()
        data = regex.sub(r'^"timestamp","message"\n?', "", data)
        data = regex.sub(r" +", "", data)
        if regex.search(r'("{4}|""[\w\d\.-]+?"")', data):
            print(data)
            data = regex.sub(r'""', '"', data)
        lines = data.split("\n")
        logs_rm.extend([
            regex.split(r"<\?xml.+?\?>", l.strip('"'))[-1]
            for l in lines if l.strip()
        ])


In [ ]:
from src.processor import  SitraProcessor
sitraproceso = SitraProcessor()
fname=Path(r"c:\Users\xiangzhou.zhang\Documents\Data\graylog\sitra\09_06_2025.csv")
# fname_ru = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Sitra\Ru_Operation_05_06.csv")
# fname_rm = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Sitra\sitran_06_06.csv")	
logs_rm  = sitraproceso.readLogFile(fname)
df_rm = sitraproceso.loadRealMovement(logs_rm)
# logs_ru = sitraproceso.readLogFile(fname_ru)
df_ru = sitraproceso.loadRUOperationRequest(logs_rm)
df_prevision = df_ru[(df_ru["Operación"] == "originChange") | (df_ru["Operación"] == "destinationChange") | (df_ru["Operación"] == "interruptionBetweenTwoPoints") | (df_ru["Operación"] == "intermediateOriginByIncidence") ]

<h2>Supresión duplicados</h2>

In [ ]:
df_supresion = df_rm[df_rm["Movimiento"] == "SUPRESIÓN"].copy()
df_supresion.sort_values(by=["FechaOrigen", "NTécnico"], inplace=True)
df_supresion["HoraPlanificada"] = df_supresion["HoraPlanificada"].apply(
    lambda x: f"{str(int(x)).zfill(6)[:2]}:{str(int(x)).zfill(6)[2:4]}:{str(int(x)).zfill(6)[4:]}"
)

In [ ]:
df_destination = df_ru[df_ru["Operación"] == "destinationChange"].copy()

In [ ]:
def filtro(grupo):
    return not (grupo["Activo"] == False).any()

## Previsión Sin supresion

In [ ]:
df_destination["Secuencia"] = df_destination["SecuenciaInicio"].astype(int)
df_supresion["Secuencia"] = df_supresion["Secuencia"].astype(int)
df_merge = pd.merge(
    df_destination,
    df_supresion,
    how="left",
    on=["FechaOrigen", "NTécnico"],
    # indicator=True,
    suffixes=('_prevision', '_supresion')
)
sin_supresion=df_merge[df_merge["Movimiento"].isna()]
df_sin_supresion= sin_supresion.groupby(["FechaOrigen", "NTécnico"]).filter(filtro)
df_sin_supresion.dropna(axis =1, how='all', inplace=True)
df_sin_supresion.head(2)

## Supresión sin previsión


In [ ]:
df_merge1 = pd.merge(
    df_destination,
    df_supresion,
    how="right",
    on=["FechaOrigen", "NTécnico"],
    # indicator=True,
    suffixes=('_desitination', '_supresion')
)

In [ ]:
df_sin_prevision = df_merge1[df_merge1["Operación"].isna()].copy()
df_sin_prevision.dropna(axis=1, how='all', inplace=True)
df_sin_prevision.head()

## Tiempo de Previsión

In [ ]:
df_merge = df_merge[df_merge["Movimiento"].notna()]
df_merge["Fecha_supresion"] = pd.to_datetime(df_merge["Fecha_supresion"])
df_merge["Fecha_prevision"] = pd.to_datetime(df_merge["Fecha_prevision"])

In [ ]:
df_merge["Tiempo_Prevision"] = df_merge["Fecha_supresion"] - df_merge["Fecha_prevision"]
df_merge["Tiempo_Prevision"] = df_merge["Tiempo_Prevision"].apply(
    lambda x: pd.to_timedelta(x, unit="m") if pd.notnull(x) else pd.NaT
)
df_merge["Tiempo_Prevision"] = df_merge["Tiempo_Prevision"].apply(
    lambda td: f"{int(td.total_seconds()//3600):02}:{int((td.total_seconds()%3600)//60):02}:{int(td.total_seconds()%60):02}" if pd.notnull(td) else None
)
df_merge.head(2)

In [ ]:
nt_producto = historico_pro.groupby(["NTécnico", "Producto"]).size().reset_index(name="conteo")
nt_ctc = historico_pro.groupby(["Código", "CTC"]).size().reset_index(name="conteo")
a = pd.DataFrame(nt_producto)
b = pd.DataFrame(nt_ctc)
b.drop(columns=["conteo"], inplace=True)
a.drop(columns=["conteo"], inplace=True)
df_final_dest = pd.merge(
    df_merge,
    a,
    how="left",
    on=["NTécnico"]
)
df_final_dest = pd.merge(
    df_final_dest,
    b,
    how="left",
    on=["Código"]
)
df_final_dest.head(1)

In [ ]:
print(df_final_dest.columns)

In [ ]:
rs = []

for _, grupo in historico_pro.groupby(["NTécnico", "FechaOrigen"]):
    grupo.reset_index(drop=True, inplace=True)
    idx_elim = grupo.index[grupo["Movimiento"] == "ELIMINACIÓN"].tolist()
    if idx_elim:
        idx = idx_elim[0]
        secuencia_elim = grupo.loc[idx, "Secuencia"]
        siguientes = grupo.loc[idx+1:]
        if not siguientes.empty and (siguientes["Secuencia"] > secuencia_elim + 3).any():
            rs.append(grupo.loc[idx+1].copy())


In [ ]:
df_rs = pd.DataFrame(rs)
df_rs.reset_index(drop=True, inplace=True)

In [ ]:
df_final_dest = df_final_dest[["FechaOrigen", "NTécnico","CTC","Producto","CódigoInicio", "SecuenciaInicio","HoraPlanificadaInicio","Activo","Razón",
                               "Fecha_prevision","FechaESB_prevision","NombreInicio","Fecha_supresion","FechaESB_supresion","Fuente","Nombre","Tiempo_Prevision"
                              ]]

In [ ]:
columns = ["FuenteMovimiento", "Vía", "TipoVía", "FuenteVía", "Retraso (segundos)", "CódigoInicio", "SecuenciaInicio","FuenteMensaje","mov_ord","Descripción"]
df_rs = df_rs.drop(columns=columns, errors='ignore')

In [ ]:
df_interruption = df_prevision[df_prevision["Operación"] == "interruptionBetweenTwoPoints"].copy()
df_interruption.head(5)

In [ ]:
df_interruption["Secuencia"] = df_interruption["SecuenciaInicio"].astype(int)
df_sin_prevision_final = pd.merge(
    df_sin_prevision,
    df_interruption,
    how="left",
    on=["NTécnico", "FechaOrigen"],
    suffixes=('_supresion', '_interruption')
)

In [ ]:
df_sin_prevision_1 = df_sin_prevision_final[df_sin_prevision_final["Operación"].isna()].copy()

In [ ]:
df_sin_prevision_1.dropna(axis=1, how='all', inplace=True)

In [ ]:
df_inter_merge = pd.merge(
    df_sin_prevision,
    df_interruption,
    how="right",
    on=["NTécnico", "FechaOrigen"],
    suffixes=('_supresion', '_interruption')
)

In [ ]:
df_sin_supresion_inter= df_inter_merge.groupby(["FechaOrigen", "NTécnico"]).filter(filtro)
df_sin_supresion_inter = df_sin_supresion_inter[df_sin_supresion_inter["Movimiento"].isna()]

In [ ]:
df_sin_supresion_inter.dropna(axis=1, how='all', inplace=True)

In [ ]:
df_sin_supresion_inter.head(2)

In [ ]:
df_inter_merge.dropna(subset=["Movimiento"], inplace=True)

In [ ]:
df_inter_merge[["FechaESB"]]

In [ ]:
df_inter_merge["FechaESB"] = pd.to_datetime(df_inter_merge["FechaESB"])
df_inter_merge["Fecha_supresion"] = pd.to_datetime(df_inter_merge["Fecha_supresion"])
df_inter_merge["Tiempo_prevision_supresión"]= df_inter_merge["Fecha_supresion"] - df_inter_merge["FechaESB"]
df_inter_merge["Tiempo_prevision_supresión"] = df_inter_merge["Tiempo_prevision_supresión"].apply(
    lambda td: f"{int(td.total_seconds()//3600):02}:{int((td.total_seconds()%3600)//60):02}:{int(td.total_seconds()%60):02}" if pd.notnull(td) else None
)

In [ ]:
df_inter_merge = pd.merge(
    df_inter_merge,
    a,
    how="left",
    on=["NTécnico"]
)
df_inter_merge = pd.merge(
    df_inter_merge,
    b,
    how="left",
    on=["Código"]
)


In [ ]:
df_inter_merge 


In [ ]:
df_inter_merge = df_inter_merge[["FechaOrigen", "NTécnico","CTC","Producto","Código", "Secuencia_supresion","HoraPlanificada","Fecha_supresion",
                                 "FechaESB_supresion","Nombre","CódigoInicio","SecuenciaInicio","HoraPlanificadaInicio","Activo","Operación","Fecha",
                               "FechaESB","CódigoFin","SecuenciaFin","NombreInicio","NombreFin","Secuencia","Tiempo_prevision_supresión"

                              ]]

In [ ]:
df_supresion[df_supresion["NTécnico"] == "82651"]

In [ ]:

df_final_dest.loc[:, "Activo"] = df_final_dest["Activo"].astype(str)
df_sin_supresion_inter.loc[:, "Activo"] = df_sin_supresion_inter["Activo"].astype(str)
df_inter_merge.loc[:, "Activo"] = df_inter_merge["Activo"].astype(str)
df_rs["FechaOrigen"] = df_rs["FechaOrigen"].dt.date
data ={"DestinoChange":df_final_dest,
       "PrevisiónSinSupresión_destino":df_sin_supresion,
       "Interrupción":df_inter_merge,  
       "PrevisiónSinSupresión_interrupt":df_sin_supresion_inter,
        "SupresiónSinPrevisión":df_sin_prevision_1,
        "RS":df_rs,
      
        
       }
today_str = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\Prevision") / f"{today_str}_Previsiones.xlsx"
guardarExcelMulti(data,fname)


In [ ]:
from pathlib import Path

directorio_Stira = Path(r"06_05_2025")
directorio_Stira = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\graylog\sitra") / str(directorio_Stira)
print(directorio_Stira)